In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Basic libraries
import sys
import numpy as np
import pickle as pk
import plotly.io as pio
import tensorflow as tf
import keras
from tensorflow.keras.models import load_model
import tree_explainer as te

pio.renderers.default = "notebook_connected"
print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

In [ ]:
# Download the model
path = 'data/calendar_weather_holiday_in_advance/'  
model_name = 'model_calendar_wather_holiday_in_advance.h5'
model = load_model(path + model_name, compile=False)
model.summary()

In [ ]:
import pickle as pk

def experiment_run(
        explain_dict, model, output_path, 
        input_path="./data/samples/", 
        X_list=["christmas", "winter", "summer"], 
        size_list=[100, 500, 1000],
        error_correction=True
        ):

    for train_size in size_list:
        for train_sample in range(1, 6):
            with open(input_path + f'X_train_sample_{train_size}_{train_sample}.pkl', 'rb') as f:
                X_train_sample = pk.load(f)
            e_mdl = te.Node(explain_dict, model, X_train_sample)
            for test_sample in X_list:
                with open(input_path + f'X_test_sample_{test_sample}.pkl', 'rb') as f:
                    X_observation = pk.load(f)
                e_mdl.explainability_tree(X_observation, err=error_correction)
                bias, matrix, errors, comp_names = e_mdl.unified_explainability()
                exec_time, model_calls = e_mdl.cum_exec_time, e_mdl.cum_model_calls
                data = {
                    'train_sample': train_sample,
                    'X': test_sample,
                    'train_size': train_size,
                    'bias': bias,
                    'matrix': matrix,
                    'errors': errors,
                    'comp_names': comp_names,
                    'exec_time': exec_time,
                    'model_calls': model_calls
                    }
                with open(output_path + f'res_{test_sample}_{train_size}_{train_sample}.pkl', 'wb') as f:
                    pk.dump(data, f)

In [ ]:
from exp_model_1 import explain_dict
output_path = "./data/results/paper/base/"

experiment_run(explain_dict, model, output_path, 
        X_list=["christmas", "winter", "summer"], 
        size_list=[100, 500, 1000])

In [ ]:
from exp_model_1_wide import explain_dict
output_path = "./data/results/paper/wide/"

experiment_run(explain_dict, model, output_path, 
        X_list=["christmas", "winter", "summer"], 
        size_list=[100, 500, 1000])

In [ ]:
from exp_model_1 import explain_dict
output_path = "./data/results/paper/biased/"

experiment_run(explain_dict, model, output_path, 
        X_list=["christmas", "winter", "summer"], 
        size_list=[100, 500, 1000], 
        error_correction=False)

In [ ]:
from exp_model_1_raw import explain_dict
output_path = "./data/results/paper/raw/"

experiment_run(explain_dict, model, output_path, 
        X_list=["summer"], 
        size_list=[100, 500, 1000])

In [ ]:
from exp_model_1 import explain_dict
output_path = "./data/results/paper/filter/"

for train_sample in range(1, 6):
    with open(f'./data/samples/X_train_sample_filter_100_{train_sample}.pkl', 'rb') as f:
        X_train_sample = pk.load(f)
    e_mdl = te.Node(explain_dict, model, X_train_sample)
    with open(f'./data/samples/X_test_sample_summer.pkl', 'rb') as f:
        X_observation = pk.load(f)
    e_mdl.explainability_tree(X_observation)
    bias, matrix, errors, comp_names = e_mdl.unified_explainability()
    exec_time, model_calls = e_mdl.cum_exec_time, e_mdl.cum_model_calls
    data = {
        'train_sample': train_sample, 'comp_names': comp_names,
        'bias': bias, 'matrix': matrix, 'errors': errors, 
        'exec_time': exec_time, 'model_calls': model_calls
    }
    with open(output_path + f'res_summer_100_{train_sample}.pkl', 'wb') as f:
                    pk.dump(data, f)

In [ ]:
import pickle as pk

def experiment_all_run(
        explain_dict, model, output_path, 
        input_path="./data/samples/",
        error_correction=True
        ):

    with open(input_path + 'X_train_sample_100_1.pkl', 'rb') as f:
        X_train_sample = pk.load(f)
    e_mdl = te.Node(explain_dict, model, X_train_sample)
    for test_sample in range(1, 51):
        with open(input_path + f'X_test_sample_{test_sample}.pkl', 'rb') as f:
            X_observation = pk.load(f)
        e_mdl.explainability_tree(X_observation, err=error_correction)
        bias, matrix, errors, comp_names = e_mdl.unified_explainability()
        exec_time, model_calls = e_mdl.cum_exec_time, e_mdl.cum_model_calls
        data = {
            'X': test_sample,
            'train_size': 100,
            'bias': bias,
            'matrix': matrix,
            'errors': errors,
            'comp_names': comp_names,
            'exec_time': exec_time,
            'model_calls': model_calls
            }
        with open(output_path + f'res_{test_sample}.pkl', 'wb') as f:
            pk.dump(data, f)

In [ ]:
from exp_model_1 import explain_dict
output_path = "./data/results/paper/base/"
experiment_all_run(explain_dict, model, output_path)

In [ ]:
from exp_model_1 import explain_dict
output_path = "./data/results/paper/biased/"
experiment_all_run(explain_dict, model, output_path, error_correction=False)

In [ ]:
from exp_model_1_wide import explain_dict
output_path = "./data/results/paper/wide/"
experiment_all_run(explain_dict, model, output_path)

In [ ]:
from exp_model_1_raw import explain_dict
output_path = "./data/results/paper/raw/"
experiment_all_run(explain_dict, model, output_path)

In [ ]:
from exp_model_1_tri import explain_dict
output_path = "./data/results/paper/tri/"
experiment_all_run(explain_dict, model, output_path)